In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
client = OpenAI(
    api_key=os.environ['api_key'],
    base_url=os.environ['base_url'],
)

In [3]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "get_weather today",
        "parameters": {
            "type": "object",
            "properties": {
                "sign": {
                    "city_name": "string",
                    "description": "get_weather name",
                },
            },
            "required": ["sign"],
        },
    },
]

In [4]:
def get_weather(sign):
    return f"get_weather {sign['city_name']},hhhh!!"

In [14]:
import json
input  = [{"role": "user", "content": "What is BEIJING weather today?"}]

response= client.responses.create(
    model="deepseek-v4-pro",
    input=input,
    tools=tools
)

input += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_weather":
            # 3. Execute the function logic for get_horoscope
            sign = json.loads(item.arguments)["sign"]
            weather = get_weather(sign)

            # 4. Provide function call results to the model
            input.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id,
                    "output": weather,
                }
            )

print("Final input:")
print(input)

#在返回给模型
response = client.responses.create(
    model="deepseek-v4-pro",
    instructions="Respond only with a weather generated by a tool.",
    tools=tools,
    input=input
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final input:
[{'role': 'user', 'content': 'What is BEIJING weather today?'}, ResponseReasoningItem(id='10165d12-945a-403c-9213-2014ee4c0cfd', summary=[], type='reasoning', content=[Content(text='The user asks about Beijing weather today. I have a get_weather tool that requires a "sign" parameter containing "city_name". I should call it with Beijing. Let me do that.', type='reasoning_text')], encrypted_content='a82aa039-c883-426c-935d-fc57d5f995e1-0', status='completed'), ResponseFunctionToolCall(arguments='{"sign": {"city_name": "Beijing"}}', call_id='call_00_3Ff2SpZ6k4IDIIPf7v9b8216', name='get_weather', type='function_call', id='46e78c24-446a-47b5-a132-277bc4f87baa', caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_00_3Ff2SpZ6k4IDIIPf7v9b8216', 'output': 'get_weather Beijing,hhhh!!'}]
Final output:
{
  "id": "2e7727bc-9b94-486a-931b-4697b7aeee25",
  "created_at": 1790235082.0,
  "error": null,
  "incomplete_details": null,
  "instruc